# 05. Clustering & Visualization

비지도 클러스터링 평가 및 시각화:
- Confusion Matrix 생성
- t-SNE 2D Embedding
- Clustering Statistics (Silhouette, Calinski-Harabasz, Davies-Bouldin)
- Ward Linkage Classification (Hungarian mapping)

## 0. 데이터 로딩 (독립 실행용)

In [ ]:
# 01번, 02번 노트북을 먼저 실행하세요 (datasets, all_results 필요)
# %run ./01_Setup_and_Data_Loading.ipynb
# %run ./02_Preprocessing_and_Evaluation.ipynb

## 1. Confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, method_name, clf_name='SVM (Linear)'):
    cm = confusion_matrix(y_true, y_pred, labels=ALL_CLASSES)
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.set_title(f'Confusion Matrix: {method_name}\n({clf_name})', fontsize=14, fontweight='bold')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(ALL_CLASSES))); ax.set_xticklabels(ALL_CLASSES, fontsize=9)
    ax.set_yticks(range(len(ALL_CLASSES))); ax.set_yticklabels(ALL_CLASSES, fontsize=9)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    for i in range(len(ALL_CLASSES)):
        for j in range(len(ALL_CLASSES)):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'cm_{method_name}.png'), dpi=150, bbox_inches='tight')
    plt.show()

print('=== Confusion Matrix 생성 (SVM Linear) ===')
for method in METHODS:
    cr = all_results[method]['classifiers']
    if 'SVM (Linear)' in cr:
        plot_confusion_matrix(cr['SVM (Linear)']['y_true'],
                              cr['SVM (Linear)']['y_pred'], method)

## 2. t-SNE 2D Embedding

In [ ]:
def plot_2d_embedding(X_reduced, y, method_name):
    tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30)
    X_2d = tsne.fit_transform(X_reduced)
    fig, ax = plt.subplots(figsize=(10, 8))
    for cls in ALL_CLASSES:
        mask = y == cls
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1], label=f'Phase {cls}', s=30, alpha=0.7)
    ax.set_title(f't-SNE 2D Embedding: {method_name}', fontsize=14, fontweight='bold')
    ax.legend(fontsize=8, loc='best', ncol=2)
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'tsne_{method_name}.png'), dpi=150, bbox_inches='tight')
    plt.show()

print('=== 2D Embedding Visualization ===')
for method in METHODS:
    plot_2d_embedding(all_results[method]['X_reduced'], all_results[method]['y'], method)

## 3. Clustering Statistics

In [ ]:
def compute_clustering_stats(X_reduced, y):
    return {
        'silhouette': silhouette_score(X_reduced, y),
        'calinski_harabasz': calinski_harabasz_score(X_reduced, y),
        'davies_bouldin': davies_bouldin_score(X_reduced, y),
    }

print('=== Clustering Statistics ===')
stats_rows = []
for method in METHODS:
    stats = compute_clustering_stats(all_results[method]['X_reduced'], all_results[method]['y'])
    stats_rows.append({'Method': method, **stats})
    print(f"  {method}: Sil={stats['silhouette']:.4f}, CH={stats['calinski_harabasz']:.1f}, "
          f"DB={stats['davies_bouldin']:.4f}")
stats_df = pd.DataFrame(stats_rows)
stats_df.to_csv(os.path.join(OUTPUT_DIR, 'clustering_stats.csv'), index=False)
print(f'\n결과 저장: {OUTPUT_DIR}/clustering_stats.csv')

## 4. Ward Linkage Classification
비지도 클러스터링 → Hungarian algorithm 매핑 → 평가

In [ ]:
from scipy.optimize import linear_sum_assignment

def cluster_to_class_mapping(y_true, cluster_labels, n_classes):
    unique_clusters = np.unique(cluster_labels)
    unique_classes = np.unique(y_true)
    size = max(len(unique_clusters), len(unique_classes))
    cost = np.zeros((size, size))
    for i, cl in enumerate(unique_clusters):
        mask = cluster_labels == cl
        for j, cls in enumerate(unique_classes):
            cost[i, j] = np.sum(mask) - np.sum(y_true[mask] == cls)
    row_ind, col_ind = linear_sum_assignment(cost)
    mapping = {}
    for r, c in zip(row_ind, col_ind):
        if r < len(unique_clusters) and c < len(unique_classes):
            mapping[unique_clusters[r]] = unique_classes[c]
    return mapping

def ward_classification(X, y, n_clusters=12):
    ward = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    cluster_labels = ward.fit_predict(X)
    mapping = cluster_to_class_mapping(y, cluster_labels, n_clusters)
    y_pred = np.array([mapping.get(cl, -1) for cl in cluster_labels])
    strict = accuracy_score(y, y_pred) * 100
    f1 = f1_score(y, y_pred, average='macro', zero_division=0) * 100
    soft = soft_accuracy_score(y, y_pred) * 100
    return {'soft_acc': soft, 'strict_acc': strict, 'f1_macro': f1,
            'y_pred': y_pred, 'cluster_labels': cluster_labels, 'mapping': mapping}

print('=== Ward Linkage Classification ===')
ward_results = {}
for method in METHODS:
    X_red = all_results[method]['X_reduced']
    y = all_results[method]['y']
    res = ward_classification(X_red, y, n_clusters=len(ALL_CLASSES))
    ward_results[method] = res
    print(f"  {method:<20s} | Soft={res['soft_acc']:.2f}% | "
          f"Strict={res['strict_acc']:.2f}% | F1={res['f1_macro']:.2f}%")

print(f"\n{'Method':<20s} {'Soft(%)':>10s} {'Strict(%)':>10s} {'F1(%)':>10s}")
print('-' * 50)
for method in METHODS:
    r = ward_results[method]
    print(f"{method:<20s} {r['soft_acc']:>10.2f} {r['strict_acc']:>10.2f} {r['f1_macro']:>10.2f}")